# Inspecting a MyPTV calibration

A calibration is usually judged by one number, the mean reprojection error,
and that number hides most of what can go wrong. A camera averaging a
respectable third of a pixel may be fine everywhere except one corner of the
frame; a set of views may be excellent apart from one whose corners were
labelled the wrong way round; and a calibration can be tight everywhere the
target went while saying nothing about the volume it never reached.

This notebook takes a finished calibration apart step by step. Every
calculation is written out rather than hidden in a helper, so that you can see
exactly what each number is. Where `myptv.makePlots.plot_calibration` has a
one-line equivalent, it is mentioned at the end of the section.

It works on **any** MyPTV calibration — points picked by hand, a target on a
translation stage, or a board moved freely — because everything comes from the
camera files and the calibration points files, which every route produces.

## 0. What we are pointing at

By default this reads the worked example in `checkerboard_moving_example`,
which has to be **run first** so that there is a calibration to look at. From
inside that folder:

```
python ../workflow.py params_file.yml moving_board_calibration
```

Change `FOLDER` below to look at any other calibration instead: it needs a
folder holding the camera files, and one holding `<camera>_cal_points`.

In [ ]:
import os

# The folder holding the camera files (cam1, cam2, ...). By default the
# worked example, which sits next to this notebook. Jupyter starts the kernel
# in the notebook's own folder, but some editors start it at the top of the
# project instead, so look in both rather than depending on which.
FOLDER = None
for candidate in ['./checkerboard_moving_example',
                  './example/checkerboard_moving_example']:
    if os.path.isdir(candidate):
        FOLDER = candidate
        break

if FOLDER is None:
    FOLDER = './checkerboard_moving_example'    # so the message below is right

# --- point this somewhere else to inspect a different calibration ---
# FOLDER = r'D:\my_experiment\calibration'

# the folder holding the calibration points files (cam1_cal_points, ...)
POINTS = os.path.join(FOLDER, 'Calibration')

CAMS = ['cam1', 'cam2', 'cam3', 'cam4']

# fail early, and say what to do about it, rather than half way down
missing = [c for c in CAMS if not os.path.exists(os.path.join(FOLDER, c))]
if missing:
    raise SystemExit(
        'No camera file for %s in %s.\\n'
        'Run the example first:\\n'
        '    cd checkerboard_moving_example\\n'
        '    python ../workflow.py params_file.yml moving_board_calibration'
        % (', '.join(missing), os.path.abspath(FOLDER)))

print('cameras   :', os.path.abspath(FOLDER))
print('points    :', os.path.abspath(POINTS))

One choice to make before starting. The `widget` backend makes the plots
interactive, which matters in section 6 where the 3D view wants rotating with
the mouse, and it needs `ipympl` (`pip install ipympl`). Without it we fall
back to `inline`, which draws the same things as static images.

In [ ]:
try:
    import ipympl                    # noqa: F401
    BACKEND = 'widget'
except ImportError:
    BACKEND = 'inline'

get_ipython().run_line_magic('matplotlib', BACKEND)
print('matplotlib backend:', BACKEND)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 5)

## 1. The camera files

A MyPTV camera file is plain text. The first line names the 3D model, and
`camera_wrapper` reads that line to decide which class to build, so the same
code below works whether the cameras are Tsai or extended Zolof.

In [ ]:
print(open(os.path.join(FOLDER, CAMS[0])).read())

In [ ]:
from myptv.imaging_mod import camera_wrapper

cams = {}
for c in CAMS:
    cw = camera_wrapper(c, FOLDER)   # tell it the name and the folder
    cw.load()                        # this reads the file and builds the model
    cams[c] = cw

for c in CAMS:
    print(f'{c}:  model = {cams[c].modelName:14s} '
          f'position = {np.round(cams[c].O, 1)}')

Two things every MyPTV camera can do, whatever its model, and the only two
this notebook needs:

* `projection(X)` — given a point in lab coordinates, where does it land on
  the sensor
* `get_r(eta, zeta)` — given a pixel, which line in space does it look along

In [ ]:
cam = cams[CAMS[0]]
X = np.array([0.0, 0.0, 1800.0])          # some point in the volume
print('a lab point            ', X)
print('lands at pixel         ', np.round(cam.projection(X), 2))
print('and the ray of that pixel has direction', np.round(cam.get_r(*cam.projection(X)), 4))

## 2. The calibration points

One row per point: the pixel it was found at, and the lab coordinates it was
assigned. `myptv.utils.Cal_image_coord` reads the first five columns.

The files written by `moving_board_calibration` carry a sixth column saying
which image the point came from. `Cal_image_coord` ignores it, so the file is
an ordinary calibration points file, but it is what lets us report the error
per view further down.

In [ ]:
path = os.path.join(POINTS, CAMS[0] + '_cal_points')
with open(path) as f:
    for k, line in enumerate(f):
        print(line.rstrip())
        if k == 2:
            break
print('...')
print('columns: eta  zeta  x_lab  y_lab  z_lab  [view]')

In [ ]:
from myptv.makePlots.plot_calibration import read_cal_points

img, lab, view = {}, {}, {}
for c in CAMS:
    img[c], lab[c], view[c] = read_cal_points(
        os.path.join(POINTS, c + '_cal_points'))
    print(f'{c}: {len(img[c])} points, '
          f'{len(np.unique(view[c])) if view[c] is not None else "?"} views')

## 3. The reprojection error, computed explicitly

This is the whole of it. For every calibration point we know

* where the corner was actually found in the image, `img`
* the lab coordinates it was assigned, `lab`

so we ask the camera model where that lab point *should* land, and compare:

$$ \mathbf{e}_i \;=\; \mathrm{projection}(\mathbf{X}_i) \;-\; \mathbf{x}_i $$

`e` is a vector in pixels: its length is the error, its direction says which
way the model is off. Everything later in this notebook is a different way of
looking at these same vectors.

In [ ]:
resid, err = {}, {}
for c in CAMS:
    # project every lab point through this camera's model
    projected = np.array([cams[c].projection(X) for X in lab[c]])

    resid[c] = projected - img[c]                 # the error vector, in px
    err[c] = np.linalg.norm(resid[c], axis=1)     # its length

print('for', CAMS[0], 'the first three points:')
for i in range(3):
    print(f'   found at {np.round(img[CAMS[0]][i], 2)}   '
          f'model says {np.round((resid[CAMS[0]][i] + img[CAMS[0]][i]), 2)}   '
          f'error {err[CAMS[0]][i]:.3f} px')

### The summary table

`mean` is the number usually quoted. Read the others too: a mean of a third of
a pixel with a maximum of three is a different calibration from one with a
maximum of half.

In [ ]:
print(f"{'camera':<8}{'points':>8}{'views':>7}{'mean':>9}{'median':>9}"
      f"{'p90':>9}{'max':>9}{'rms':>9}")
for c in CAMS:
    e = err[c]
    nv = len(np.unique(view[c])) if view[c] is not None else 0
    print(f'{c:<8}{len(e):8d}{nv:7d}{e.mean():9.4f}{np.median(e):9.4f}'
          f'{np.percentile(e, 90):9.4f}{e.max():9.4f}'
          f'{np.sqrt((e**2).mean()):9.4f}')
print('\nall in pixels')

# the same thing in one line:
#   from myptv.makePlots.plot_calibration import calibration_report
#   calibration_report.from_folder(POINTS, FOLDER, CAMS).summary()

In [ ]:
# how the error is distributed, one histogram per camera
fig, ax = plt.subplots()
edges = np.linspace(0, max(np.percentile(err[c], 99.5) for c in CAMS), 40)
for c in CAMS:
    ax.hist(err[c], bins=edges, histtype='step', lw=1.6, label=c)
    ax.axvline(err[c].mean(), ls=':', lw=1.0, alpha=0.6)
ax.set_xlabel('reprojection error [px]')
ax.set_ylabel('points')
ax.set_title('error distribution by camera (dotted = means)')
ax.legend();

## 4. The error of each view

This is the plot to look at first, and the counterpart of MATLAB's
`showReprojectionErrors`. Each bar is one image.

The calculation is just a grouping: take the errors of one camera, split them
by the view column, and average each group.

* **one view far above its neighbours** — almost always an image whose corners
  were labelled the wrong way round. Remove it and calibrate again.
* **a group of neighbouring views high together** — something else; look at
  those images.

In [ ]:
per_view = {}
for c in CAMS:
    per_view[c] = {}
    for v in np.unique(view[c]):
        mask = view[c] == v                 # the points belonging to this view
        per_view[c][int(v)] = err[c][mask].mean()

# the worst ten, over all cameras
rows = [(c, v, m) for c in CAMS for v, m in per_view[c].items()]
rows.sort(key=lambda r: -r[2])
print('the worst views:')
for c, v, m in rows[:10]:
    print(f'   {c:<8} view {v:<4} {m:.4f} px')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
all_views = sorted({v for c in CAMS for v in per_view[c]})
width = 0.8/len(CAMS)

for k, c in enumerate(CAMS):
    xs = [all_views.index(v) + k*width - 0.4 + width/2 for v in per_view[c]]
    ys = list(per_view[c].values())
    ax.bar(xs, ys, width=width, label=c)

overall = np.mean([m for _, _, m in rows])
ax.axhline(overall, color='k', ls='--', lw=1, label=f'mean {overall:.3f} px')
ax.set_xticks(range(len(all_views)))
ax.set_xticklabels(all_views, fontsize=7, rotation=90)
ax.set_xlabel('view'); ax.set_ylabel('mean error [px]')
ax.set_title('mean reprojection error per view')
ax.legend(fontsize=8);

## 5. Where in the frame the error lives

Each arrow starts at a calibration point and points the way the model is off,
exaggerated so it can be seen. This is the plot that distinguishes *noise*
from *a model that does not fit*:

* arrows pointing every which way — measurement noise, which is what a good
  calibration looks like
* arrows agreeing with their neighbours, growing towards the edges — lens
  distortion the model is not capturing
* one patch of agreeing arrows in an otherwise random field — usually a
  misplaced point

In [ ]:
def resolution(cam):
    '''the sensor size of a camera, if its model records one'''
    r = getattr(cam.camera, 'resolution', None)
    if r is None:                  # some models do not store it
        return None
    return float(r[0]), float(r[1])


CAM = CAMS[0]                     # change this to look at another camera
W, H = resolution(cams[CAM])
print(f'{CAM} is {W:.0f} x {H:.0f} px')

p = img[CAM]
d = resid[CAM]
e = err[CAM]

# scale the arrows so a typical one is a few percent of the frame
scale = 0.06*max(W, H)/max(np.median(e), 1e-9)

fig, ax = plt.subplots(figsize=(9, 6))
q = ax.quiver(p[:, 0], p[:, 1], d[:, 0]*scale, d[:, 1]*scale, e,
              angles='xy', scale_units='xy', scale=1.0, cmap='viridis',
              width=0.003)
plt.colorbar(q, ax=ax, label='error [px]')
ax.add_patch(plt.Rectangle((0, 0), W, H, fill=False, ec='0.6'))
ax.set_xlim(-0.02*W, 1.02*W); ax.set_ylim(1.02*H, -0.02*H)
ax.set_aspect('equal')
ax.set_xlabel('eta [px]'); ax.set_ylabel('zeta [px]')
ax.set_title(f'{CAM}: error over the frame, arrows exaggerated {scale:.0f}x');

### Which part of the frame was calibrated at all

Outside the region the points cover, the camera model is extrapolating, and
the error it shows on the points it was fitted to says nothing about how it
behaves there. If your particles will appear where the target never went, the
calibration does not really cover them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for c in CAMS:
    ax.plot(img[c][:, 0], img[c][:, 1], '.', ms=2, alpha=0.5, label=c)
ax.add_patch(plt.Rectangle((0, 0), W, H, fill=False, ec='k', lw=1.2))
ax.set_xlim(-0.03*W, 1.03*W); ax.set_ylim(1.03*H, -0.03*H)
ax.set_aspect('equal')
ax.set_xlabel('eta [px]'); ax.set_ylabel('zeta [px]')
ax.set_title('where the calibration points are in the frame')
ax.legend(markerscale=4, fontsize=8);

## 6. Where the cameras are

The counterpart of MATLAB's `showExtrinsics`. Two things to check: that the
arrangement is the arrangement of your actual rig, which catches a camera
reconstructed on the wrong side or pointing the wrong way; and that the cloud
of calibration points covers your measurement volume.

The field of view is drawn by asking each camera, through `get_r`, which way
it looks at each corner of its sensor. That works for any 3D model.

One subtlety, written out because it is easy to trip over. `get_r` returns the
direction of a **line**, and the sign of that direction is not fixed by
anything: the line through the camera along `r` is the same line as the one
along `-r`, so `img_system.stereo_match`, which measures distances between
lines, cannot care. It matters here, though, because a field of view drawn
along the wrong sign points away from the scene. So we settle the sign from
the data: the calibration points are in front of the camera by definition.

If `BACKEND` came out as `widget` above, this plot can be rotated with the
mouse; if it came out as `inline` it is a static image, and installing
`ipympl` will make it interactive.

In [ ]:
def frustum(cam, seen, length):
    '''
    The four corners of a camera's field of view, at a given distance.

    cam    - a camera_wrapper
    seen   - the lab points this camera saw, used only to settle the sign
    length - how far out to draw it, in lab units
    '''
    W, H = resolution(cam)
    O = np.asarray(cam.O, float)

    # the direction the camera looks in at each corner of its sensor
    rays = []
    for (u, v) in [(0, 0), (W, 0), (W, H), (0, H)]:
        r = np.asarray(cam.get_r(u, v), float)
        rays.append(r/np.linalg.norm(r))
    rays = np.array(rays)

    # settle the sign: the points this camera saw are in front of it
    towards = seen.mean(axis=0) - O
    towards = towards/np.linalg.norm(towards)
    if rays.mean(axis=0).dot(towards) < 0:
        rays = -rays

    return O, O + length*rays


fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection='3d')

# the calibration points, as a grey cloud
allX = np.concatenate([lab[c] for c in CAMS])
ax.plot(allX[:, 0], allX[:, 1], allX[:, 2], '.', ms=1, color='0.55', alpha=0.4)

centre = allX.mean(axis=0)
length = 0.75*np.median([np.linalg.norm(np.asarray(cams[c].O) - centre)
                         for c in CAMS])
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for k, c_name in enumerate(CAMS):
    col = colors[k % len(colors)]
    O, Q = frustum(cams[c_name], lab[c_name], length)
    ax.plot([O[0]], [O[1]], [O[2]], 'o', color=col, ms=9)
    ax.text(O[0], O[1], O[2], '  ' + c_name, color=col)
    for i in range(4):                       # the edges from the camera out
        ax.plot(*zip(O, Q[i]), '-', color=col, lw=0.8, alpha=0.8)
    for i in range(4):                       # the rectangle at the far end
        ax.plot(*zip(Q[i], Q[(i + 1) % 4]), '-', color=col, lw=1.2)

# equal aspect, or the geometry looks wrong
lims = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
span = (lims[:, 1] - lims[:, 0]).max()/2
mid = lims.mean(axis=1)
ax.set_xlim3d(mid[0]-span, mid[0]+span)
ax.set_ylim3d(mid[1]-span, mid[1]+span)
ax.set_zlim3d(mid[2]-span, mid[2]+span)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('the cameras and the calibration target');

### The distances between the cameras

These do not depend on the choice of lab frame, so if you know any of them
from the rig — two cameras on a common mount, say — comparing is a genuinely
independent check on the calibration, not a restatement of the fit.

In [ ]:
import itertools
for a, b in itertools.combinations(CAMS, 2):
    d = np.linalg.norm(np.asarray(cams[a].O) - np.asarray(cams[b].O))
    print(f'{a} - {b}: {d:9.2f}')

## 7. Do the cameras agree with each other?

Everything so far asks whether each camera fits its own points. This asks the
harder question, and the one the calibration exists to answer: take the same
physical corner as seen by several cameras, and do their rays actually meet?

`img_system.stereo_match` takes a pixel from each camera, builds the ray of
each, and returns the point closest to all of them together with how far apart
they were at that point. That distance is in the units of your lab
coordinates, and it is the real measure of a multi-camera calibration.

In [ ]:
from myptv.imaging_mod import img_system

system = img_system([cams[c] for c in CAMS])

# Gather each physical corner: the same lab coordinate, seen by several
# cameras. The lab coordinate is what identifies it, since every camera that
# saw a given corner was given the same one. Keep the view it came from too,
# which the next cell needs.
corner, whichview = {}, {}
for i, c in enumerate(CAMS):
    for x, X, v in zip(img[c], lab[c], view[c]):
        key = tuple(np.round(X, 4))
        corner.setdefault(key, {})[i] = tuple(x)
        whichview[key] = int(v)

shared = [k for k, v in corner.items() if len(v) >= 2]
print(f'{len(shared)} corners were seen by two or more cameras')

gap, pos, pos_view = [], [], []
for k in shared:
    out = system.stereo_match(corner[k], 1e9)   # 1e9 = accept any distance
    if out is None:
        continue
    X, used, d = out
    gap.append(d)
    pos.append(X)
    pos_view.append(whichview[k])

gap = np.array(gap)
pos = np.array(pos)
pos_view = np.array(pos_view)

print(f'\nthe rays meet to within:')
print(f'   median {np.median(gap):.4f}   p90 {np.percentile(gap, 90):.4f}'
      f'   max {gap.max():.4f}   (lab units)')

In [ ]:
fig, ax = plt.subplots()
ax.hist(gap, bins=60)
ax.set_xlabel('distance between the rays where they come closest [lab units]')
ax.set_ylabel('corners')
ax.set_title('do the cameras agree?');

### The board's own size, measured back out

A last check that owes nothing to any single camera. The corners we just
triangulated came from a board whose squares are of known size. Measuring the
distance from each triangulated corner to its nearest neighbour should give
that size back.

This is not a free test of the *scale*, which was set by declaring the square
size when calibrating. What it tests is everything downstream: the adjustment,
the calibration points, the fitted camera model and the triangulation above,
all of which have to be consistent for the number to come out right.

In [ ]:
from scipy.spatial import cKDTree

SQUARE = 15.0        # the true spacing of the corners, in lab units

# One view at a time. The board was somewhere different in each of them, and
# two boards can pass closer to one another than 15 mm, so a nearest
# neighbour taken over the whole cloud at once would sometimes land on a
# corner of a different view and give an answer that is too small.
spacing = []
for v in np.unique(pos_view):
    P = pos[pos_view == v]
    d, _ = cKDTree(P).query(P, k=2)     # k=2 because the nearest is itself
    spacing.append(d[:, 1])
spacing = np.concatenate(spacing)

print(f'recovered spacing {spacing.mean():.4f} +- {spacing.std():.4f}')
print(f'true spacing      {SQUARE}')
print(f'error             {1000*(spacing.mean() - SQUARE):+.0f} um '
      f'({100*(spacing.mean() - SQUARE)/SQUARE:+.3f} %)')

## 8. What would dropping a bad view do?

If section 4 showed one view far worse than the rest, this says what the
numbers would look like without it. It does **not** recalibrate — for that,
remove the image and run the calibration again — but it tells you whether it
is worth doing.

In [ ]:
DROP = []          # e.g. [3] to see the effect of dropping view 3

if not DROP:
    print('set DROP to a list of view numbers to try this')
else:
    for c in CAMS:
        keep = ~np.isin(view[c], DROP)
        print(f'{c}: mean {err[c].mean():.4f} -> {err[c][keep].mean():.4f} px'
              f'   ({(~keep).sum()} of {len(keep)} points dropped)')

---

## The same thing in one line

Everything above is available from `calibration_report`, which is what to use
once the steps are familiar:

```python
from myptv.makePlots.plot_calibration import calibration_report

rep = calibration_report.from_folder(POINTS, FOLDER, CAMS)
rep.summary()                    # section 3
rep.per_view(printout=True)      # section 4
rep.plot_error_per_camera()      # section 3
rep.plot_error_per_view()        # section 4
rep.plot_error_over_frame('cam1')# section 5
rep.plot_coverage()              # section 5
rep.plot_extrinsics()            # section 6
```